# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [1]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

Python 3.12.4 (main, Mar  5 2026, 19:12:10) [GCC 13.3.0]
asyncio version: built-in


## Sección 1: await secuencial vs asyncio.gather — la diferencia central

**Qué vamos a ver:** la diferencia entre M2 (await secuencial) y M4 (gather) es literalmente una línea de código. Los tiempos medidos hacen que la diferencia sea imposible de ignorar.

**Predicción del modelo:**
- M2 (await en secuencia): `T_total = N × T_tarea` — cada usuario espera a que el anterior termine
- M4 (gather): `T_total ≈ T_tarea_más_lenta` — todos los usuarios esperan *al mismo tiempo*

Con N=5 tareas de 1.0s cada una:
- M2 debería tardar: **5.0s**
- M4 debería tardar: **~1.0s**

Corre las celdas y verifica. ¿Coincide con la predicción? ¿El speedup es exactamente 5×, o hay overhead?

> Referencia: `04a_asyncio_fundamentos.md` — sección "asyncio.gather — M4 en una línea"

In [2]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

=== M2: await secuencial ===
  τ1: 1.01s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M2: 5.01s  (esperado: 5.0s = N×T)



`await` le dice al event loop "puedes hacer otra cosa mientras espero" — pero si no hay ninguna otra tarea registrada, el event loop no tiene nada más que hacer y simplemente espera.

Es como un mesero que le dice a la cocina "avísame cuando esté listo el platillo 1" — pero si solo hay una mesa y un platillo pedido, de todas formas se queda parado esperando aunque técnicamente "podría" atender otras mesas.

```
# Lo que pasa con el for:
event loop: "voy a esperar tarea1..."
            (no hay nadie más)
            "...lista, voy a esperar tarea2..."
            (no hay nadie más)
            "...lista, voy a esperar tarea3..."
```

```
# Lo que pasa con gather():
event loop: "voy a esperar tarea1, tarea2, tarea3 al mismo tiempo"
            (las tres duermen juntas)
            "...todas listas"
```

La diferencia es que `gather()` registra todas las tareas en el event loop **antes** de empezar a esperar, entonces cuando una hace `await` y se pausa, el event loop tiene otras dos para atender. Ahí sí se aprovecha el mecanismo.

In [3]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

=== M4: asyncio.gather ===
  τ1: 1.00s
  τ2: 1.00s
  τ3: 1.00s
  τ4: 1.00s
  τ5: 1.00s
Tiempo total M4: 1.00s  (esperado: ~1.0s = T_max)

Speedup M4/M2: 5.0x

Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅
Las 5 tareas de 1.0s corren en ~1.0s en lugar de 5.0s


`gather` hace dos cosas:

**1. Registra todas las tareas en el event loop al mismo tiempo** — antes de que ninguna empiece a esperar, ya están todas "en la lista".

**2. Espera a que todas terminen** y regresa los resultados en orden.

El `*[tarea_io(...) for i in range(N_TAREAS)]` simplemente crea las 5 tareas y las pasa todas juntas a `gather` con el `*` que las desempaca.

---

Lo que pasa internamente:

```
t=0s  gather registra tarea1, tarea2, tarea3, tarea4, tarea5
      tarea1 → await → se pausa, event loop pasa a tarea2
      tarea2 → await → se pausa, event loop pasa a tarea3
      tarea3 → await → se pausa, event loop pasa a tarea4
      tarea4 → await → se pausa, event loop pasa a tarea5
      tarea5 → await → se pausa
      (todas durmiendo al mismo tiempo)
t=1s  tarea1 despierta, tarea2 despierta... todas terminan
Total: ~1s
```

La clave es que el event loop tiene 5 tareas pausadas simultáneamente esperando su `asyncio.sleep`. Como todas están dormidas al mismo tiempo, cuando pasa 1 segundo todas despiertan juntas.

Es como el mesero que toma el pedido de las 5 mesas antes de ir a la cocina — todas las órdenes están en la cocina al mismo tiempo, en lugar de esperar a que salga una antes de tomar la siguiente.

El flujo es:

```
gather recibe: tarea1, tarea2, tarea3, tarea4, tarea5

tarea1 corre hasta await → se pausa → event loop pasa a tarea2
tarea2 corre hasta await → se pausa → event loop pasa a tarea3
tarea3 corre hasta await → se pausa → event loop pasa a tarea4
tarea4 corre hasta await → se pausa → event loop pasa a tarea5
tarea5 corre hasta await → se pausa

(todas dormidas al mismo tiempo)

1 segundo después...

tarea1 despierta → termina
tarea2 despierta → termina
tarea3 despierta → termina
tarea4 despierta → termina
tarea5 despierta → termina
```

El momento clave es ese — todas entran a su `await asyncio.sleep` casi al mismo tiempo porque el event loop las lanza una tras otra muy rápido (microsegundos). Entonces todas están durmiendo en paralelo y todas despiertan al mismo tiempo después de 1 segundo.

Por eso el total es ~1s en lugar de 5s.

## Sección 2: Traza del event loop con asyncio debug mode

**Qué vamos a ver:** asyncio tiene un modo de depuración que emite advertencias cuando el event loop se bloquea más tiempo del esperado. Es la herramienta para diagnosticar el anti-patrón de `time.sleep` dentro de funciones `async`.

**El problema a demostrar:**
```
time.sleep(0.3)       ← bloquea el hilo del OS durante 300ms
                         → el event loop no puede ejecutar NINGUNA otra coroutine
                         → gather con 3 tareas de 0.3s tarda 0.9s (secuencial)

await asyncio.sleep(0.3)  ← registra un callback y cede el control
                              → el event loop puede ejecutar otras coroutines
                              → gather con 3 tareas de 0.3s tarda ~0.3s (M4)
```

**Predicción:**
- `gather` con `asyncio.sleep(0.3)`: debería tardar **~0.3s**
- `gather` con `time.sleep(0.3)`: debería tardar **~0.9s** (sin mejora — M1)

El modo debug (`loop.set_debug(True)`) detectará el bloqueo y emitirá una advertencia en el segundo caso. Observa el output completo.

> Referencia: `04a_asyncio_fundamentos.md` — sección "time.sleep vs asyncio.sleep"

In [4]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

=== gather con tareas CORRECTAS (asyncio.sleep) ===
  A: inicio
  B: inicio
  C: inicio
  A: fin
  B: fin
  C: fin
Tiempo: 0.33s  (esperado: ~0.3s)

=== gather con tareas BLOQUEANTES (time.sleep) ===
  X: inicio
  X: fin
  Y: inicio
  Y: fin
  Z: inicio
  Z: fin
Tiempo: 0.91s  (esperado: ~0.9s — sin mejora)

Observa: con time.sleep, gather NO ayuda.
time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.


In [5]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

**TAREA — implementación guiada**

Esta sección te pide implementar dos modelos *incorrectos* para CPU-bound e ineficientes para sus casos de uso, y medir por qué fallan la promesa de concurrencia/paralelismo.

**TAREA 3.1 — M2: async con await secuencial**

Ya viste M2 en la Sección 1. Ahora explícalo formalmente:
- ¿Qué condición de M4 (`exec(τⱼ) ∩ wait(τᵢ) ≠ ∅`) falla en M2?
- ¿En qué se diferencia `await fn1(); await fn2()` de `asyncio.gather(fn1(), fn2())`?

Escribe la respuesta como comentario en la celda antes de correr el código.

**TAREA 3.2 — M3: threading CPU-bound**

El GIL (Global Interpreter Lock) garantiza que solo un hilo ejecuta bytecode Python a la vez. Para tareas CPU-bound (`wait(τᵢ) = ∅`), el GIL nunca se libera — el threading no puede producir paralelismo real.

**Predicción:** con N=4 hilos en una tarea CPU-bound, el speedup esperado es **≈1×** (sin mejora). Puede incluso ser < 1× por el overhead de sincronización del GIL.

Implementa el threading y mide. ¿Coincide con la predicción?

In [ ]:
# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
#Porque no aprovecha las esperas que se realizan en cada tarea, ejecuta todo de una forma secuencial
# No permite que otras tareas se realicen mientras que esperas entonces el tiempo es el mismo que si primero definieras y realizaras
# Una tarea y despues fueras con la otra.
# Responde con la definición formal: ¿qué condición de M4 falta en M2?
# M4 tiene condiciones de: wait != ∅, la cual se cumple en M2, pero la que no se cumple es exec(τⱼ) ∩ wait(τᵢ) ≠ ∅, es decir, que la ejecución de una tarea se superponga con la espera de otra tarea. 
# En M2, cada tarea se ejecuta secuencialmente, por lo que no hay superposición entre la ejecución de una tarea y la espera de otra, lo que resulta en un tiempo total igual a la suma de los tiempos individuales de cada tarea.
# Pregunta: ¿en qué se diferencia await fn1(); await fn2() de await asyncio.gather(fn1(), fn2())?
# La diferencia es que en el asyncio gather se definen todas las tareas a ejecutar desde el inicio y el event loop 
# gestiona las tareas de forma que se aprovechen los tiempos de espera, mientras que con await secuencialmente, cada tarea se ejecuta una después de la otra sin aprovechar las esperas, lo que resulta en un tiempo total mayor.

# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# --- Threading M3 ---
# TODO: implementa con threading.Thread y mide el tiempo
# Pista: usa la misma tarea_cpu_bound con N_HILOS hilos
t0 = time.perf_counter()
hilos = []
for i in range (N_HILOS):
    h = threading.Thread(target=tarea_cpu_bound, args=(N_CPU,))
    hilos.append(h)
for h in hilos:
    h.start()
for h in hilos:
    h.join() #bloquea el programa principal hasta que los hilos terminen, lo que hace que no midas el tiempo hasta que todos terminen
t_threading = time.perf_counter() - t0


# TODO: imprime los tiempos y el speedup
# ¿Qué dice el resultado sobre M3 + GIL en Python?
# El resultado obtenido muestra un speedup lo cual es un poco curioso porque no deberia de haber mejora ya que se mantiene el mismo tiempo, 
# esto se debe a que el GIL en Python impide que múltiples hilos ejecuten código Python al mismo tiempo, lo que significa que incluso con múltiples hilos, solo uno puede ejecutar código Python a la vez. 
# Sin embargo, en este caso específico, es posible que el sistema operativo esté asignando los hilos de manera eficiente o que la tarea CPU-bound no esté completamente bloqueada por el GIL, lo que podría explicar el speedup observado. 

print(f'M1 secuencial: {t_secuencial:.2f}s')
print(f'M3 threading: {t_threading:.2f}s')
print(f'Speedup: {t_secuencial / t_threading:.2f}')

M1 secuencial: 2.55s
M3 threading: 2.08s
Speedup: 1.23


## Sección 4: Race condition reproducible + fix con Lock

**TAREA — reproducir y corregir**

Las condiciones de carrera (race conditions) son el bug clásico de la concurrencia con memoria compartida. Ocurren cuando múltiples hilos leen y escriben la misma variable sin coordinación.

**Por qué ocurre:**
La operación `contador += 1` parece atómica pero en realidad son 3 pasos:
```
LOAD  contador      → lee el valor actual al registro de la CPU
ADD   registro, 1   → incrementa en 1
STORE registro → contador  → escribe de vuelta
```
Si dos hilos ejecutan esto concurrentemente, el segundo puede leer el valor *antes* de que el primero escriba su resultado — se pierde un incremento.

**Predicción:**
- Sin lock: `contador` final < `N_INCREMENTOS × N_HILOS` (incrementos perdidos, no determinístico)
- Con `threading.Lock`: `contador` final = `N_INCREMENTOS × N_HILOS` siempre

Corre varias veces sin lock. ¿El resultado es siempre distinto? ¿Siempre menor que el esperado?

> Nota: el GIL de Python reduce (no elimina) las race conditions. Este ejemplo las reproduce porque el increment no es atómico incluso con el GIL.

In [ ]:
import threading

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1   # NO atómico: LOAD, ADD, STORE separados

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

# TAREA 4.2 — Fix con Lock
# TODO: implementa incrementar_con_lock usando threading.Lock
contador_con_lock = [0]
lock = threading.Lock()

def incrementar_con_lock():
    for _ in range (N_INCREMENTOS):
        with lock:
            contador_con_lock[0] += 1   # ATÓMICO: el lock asegura exclusión mutua

hilos = [threading.Thread(target=incrementar_con_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()
# El resultado debe ser siempre exactamente N_INCREMENTOS × N_HILOS_RACE

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print(f'Con Lock - esperado: {esperado:,}, obtenido: {contador_con_lock[0]:,}')
print(f'Diferencia: {esperado - contador_con_lock[0]:,} incrementos perdidos')

# TODO: verifica que contador_con_lock == esperado
# Si se cumple, pero el contador_sin_lock igual salio con 0 eso puede deberse a que la prueba es muy pequeña o algo así.

Sin lock  — esperado: 400,000, obtenido: 400,000
Diferencia: 0 incrementos perdidos
Con Lock - esperado: 400,000, obtenido: 400,000
Diferencia: 0 incrementos perdidos


## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

**TAREA — implementación completa**

Implementa el servidor chatbot v2 del Escenario A: LLM como API remota, I/O-bound. Compara la versión secuencial (v1) con la concurrente (v2).

**Arquitectura del chatbot v2:**
```
N usuarios simultáneos
        │
asyncio.gather(handle_request(0), handle_request(1), ..., handle_request(N-1))
        │
  ┌─────┴──────────────────────────────────────────┐
  │  Event loop (1 hilo)                           │
  │                                                │
  │  τ_u0 exec → await BD(50ms) → await LLM(1.5s) │
  │    τ_u1 exec → await BD(50ms) → await LLM(1.5s)│
  │      τ_u2 exec → await BD(50ms) → await LLM...│
  └──────────────────┬─────────────────────────────┘
                     │ (simultáneamente)
            [BD asyncpg]  [LLM API aiohttp]
```

**Predicciones:**
- v1 (secuencial) con N=10: `T_total ≈ 10 × 1.55s = 15.5s`
- v2 (gather) con N=10: `T_total ≈ 1.55s`
- Latencia del usuario 10 en v2: **similar a la del usuario 1** (todos esperan en paralelo)

Implementa `servidor_v1` y `servidor_v2`, mide con N=10, y responde:
1. ¿La latencia es uniforme entre usuarios en v2? ¿Por qué?
2. ¿Qué pasaría si uno de los usuarios tuviera `time.sleep` en su handler?

> Referencia: `04a_asyncio_fundamentos.md` — sección "Chatbot v2"

In [ ]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(τᵢ): I/O a base de datos — ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(τᵢ): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    # exec(τᵢ)
    t_inicio = time.perf_counter()

    # wait(τᵢ): BD
    historial = await consultar_bd(user_id)

    # wait(τᵢ): LLM
    respuesta = await llamar_llm(historial)

    # exec(τᵢ)
    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# TAREA 5.1 — Servidor secuencial (chatbot v1 como baseline)
async def servidor_v1(n_usuarios: int):
    """M1: un usuario a la vez"""
    # TODO: implementa el servidor secuencial
    # Hint: un for loop con await handle_request(i) para cada usuario
    t_inicio = time.perf_counter()
    resultados = []
    for _ in range(n_usuarios):
        r = await handle_request(_)
        resultados.append(r)
    latencia = time.perf_counter() - t_inicio
    return resultados, latencia

# TAREA 5.2 — Servidor concurrente (chatbot v2)
async def servidor_v2(n_usuarios: int):
    """M4: todos los usuarios concurrentes con gather"""
    # TODO: implementa con asyncio.gather
    t_inicio = time.perf_counter()
    resultados = await asyncio.gather(
        *[handle_request(i) for i in range(n_usuarios)]
    )
    latencia = time.perf_counter() - t_inicio
    return resultados, latencia

# TAREA 5.3 — Compara v1 vs v2 con N=10 usuarios
# Mide tiempos totales y latencias promedio
# ¿Cuántas veces más rápido es v2?
# ¿La latencia de cada usuario es similar en v2? ¿Por qué?


N = 10

resultados_v1, latencia_v1 = await servidor_v1(N)
resultados_v2, latencia_v2 = await servidor_v2(N)
time_v1 = latencia_v1
time_v2 = latencia_v2
latencia_promedio_v1 = sum(r['latencia'] for r in resultados_v1) / len(resultados_v1)
latencia_promedio_v2 = sum(r['latencia'] for r in resultados_v2) / len(resultados_v2)
speedup = time_v1 / time_v2


print(f'Comparando v1 vs v2 con {N} usuarios...')
print('TODO: implementa servidor_v1 y servidor_v2 arriba')

Comparando v1 vs v2 con 10 usuarios...
TODO: implementa servidor_v1 y servidor_v2 arriba
